Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')

Load the data

In [36]:
from ssegnn.dataloader import DataLoader
from ssegnn.config import load_config

config = load_config('configs/data.yaml')

dataloader = DataLoader(
    label_path=config['paths']['label'],
    data_path=config['paths']['data'],
    start_date=config['data']['start_date'],
    end_date=config['data']['end_date'],
    download=config['data']['download'],
    load=config['data']['load'],
    verbose=True,
)

Number of sites: 70
Start date: 2020-01-01
End date: 2024-12-31




100%|██████████| 70/70 [00:33<00:00,  2.12it/s]


Saving data to file...
Done!
Loading data from file...
Done!


Create a temporal graph with node features and targets

In [78]:
from ssegnn.config import load_config

hyper = load_config('configs/hyper.yaml')

data = dataloader.get_graph(k=hyper['graph']['k'],
                            r=hyper['graph']['r']
                            )

Building k-NN graph with k=4...
Done!


Number of nodes: 69
Number of node features: 3
Number of snapshots: 1802
Number of edges: 332




Normalize data temporally

In [45]:
# Apply normalization transform
from ssegnn.graph import TemporalGraphNormalize

normalize_transform = TemporalGraphNormalize(normalize=True, fill_nan='both')
data = normalize_transform(data)

Transform the temporal graph into a graph with static features from (a) the signature, and (b) random features

In [75]:
from ssegnn.signature import SignatureFeatures, RandomFeatures
from ssegnn.config import load_config

hyper = load_config('configs/hyper.yaml')

# Configure the signature transform as needed
sig_transform = SignatureFeatures(
    sig_depth=hyper['sig']['depth'],         # or your desired depth
    normalize=hyper['sig']['normalize'],      # whether to normalize the signature features
    log_signature=hyper['sig']['log_signature'], # use log-signature or not
    time_augment=hyper['sig']['time_augment'],  # add time as a feature or not
    lead_lag=hyper['sig']['lead_lag']      # use lead-lag augmentation or not
)

# Apply the transform to your temporal graph data
static_graph = sig_transform(data)

In [76]:
static_graph.x.shape

torch.Size([69, 121])

Train a GCN classifier to predict labels from signature features

In [77]:
import torch
from ssegnn.graph import NodeSplitMask
from ssegnn.model import ClassifierGCN


hyper = load_config('configs/hyper.yaml')

split_transform = NodeSplitMask(train_ratio=hyper['split']['train_ratio'],
                                val_ratio=hyper['split']['val_ratio'],
                                test_ratio=hyper['split']['test_ratio'],
                                seed=hyper['split']['seed'])
static_graph = split_transform(static_graph)


model = ClassifierGCN(node_features=static_graph.num_node_features,
                      hidden_features=hyper['hidden_features'],
                      num_classes=2)

optimizer = torch.optim.Adam(model.parameters(),
                             lr=hyper['lr'],
                             weight_decay=hyper['weight_decay'])
criterion = torch.nn.CrossEntropyLoss()

train_losses = []
val_losses = []

# Training loop
epochs = hyper
for epoch in range(int(hyper['num_epochs'])):
    model.train()
    optimizer.zero_grad()
    out = model(static_graph.x, static_graph.edge_index, static_graph.edge_attr)
    # Only use labeled nodes for training
    loss = criterion(out[static_graph.train_mask], static_graph.y[static_graph.train_mask].long())
    train_losses.append(loss.item())
    loss.backward()
    optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        pred = out.argmax(dim=1)
        train_acc = (pred[static_graph.train_mask] == static_graph.y[static_graph.train_mask]).float().mean().item()
        val_acc = (pred[static_graph.val_mask] == static_graph.y[static_graph.val_mask]).float().mean().item() if static_graph.val_mask.sum() > 0 else float('nan')
    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f} | Train Acc: {train_acc:.4f}")

# Test evaluation
model.eval()
with torch.no_grad():
    out = model(static_graph.x, static_graph.edge_index, static_graph.edge_attr)
    test_loss = criterion(out[static_graph.test_mask], static_graph.y[static_graph.test_mask].long()).item()
    pred = out.argmax(dim=1)
    test_acc = (pred[static_graph.test_mask] == static_graph.y[static_graph.test_mask]).float().mean().item() if static_graph.test_mask.sum() > 0 else float('nan')
print(f"Test Accuracy: {test_acc:.4f}")

Epoch 000 | Loss: nan | Train Acc: 0.8545
Epoch 010 | Loss: nan | Train Acc: 0.8545
Epoch 020 | Loss: nan | Train Acc: 0.8545
Epoch 030 | Loss: nan | Train Acc: 0.8545
Epoch 040 | Loss: nan | Train Acc: 0.8545
Epoch 050 | Loss: nan | Train Acc: 0.8545
Epoch 060 | Loss: nan | Train Acc: 0.8545
Epoch 070 | Loss: nan | Train Acc: 0.8545
Epoch 080 | Loss: nan | Train Acc: 0.8545
Epoch 090 | Loss: nan | Train Acc: 0.8545
Epoch 100 | Loss: nan | Train Acc: 0.8545
Epoch 110 | Loss: nan | Train Acc: 0.8545
Epoch 120 | Loss: nan | Train Acc: 0.8545
Epoch 130 | Loss: nan | Train Acc: 0.8545
Epoch 140 | Loss: nan | Train Acc: 0.8545
Epoch 150 | Loss: nan | Train Acc: 0.8545
Epoch 160 | Loss: nan | Train Acc: 0.8545
Epoch 170 | Loss: nan | Train Acc: 0.8545
Epoch 180 | Loss: nan | Train Acc: 0.8545
Epoch 190 | Loss: nan | Train Acc: 0.8545
Epoch 200 | Loss: nan | Train Acc: 0.8545
Epoch 210 | Loss: nan | Train Acc: 0.8545
Epoch 220 | Loss: nan | Train Acc: 0.8545
Epoch 230 | Loss: nan | Train Acc: